In [1]:
# Set up all required functions
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import dense_rank, desc,when
from pyspark.sql.functions import dense_rank, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType,CharType,ArrayType
from itertools import chain

In [2]:
# Initialize system configurations

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.8-hotspot"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Verify BEFORE creating the session

print("PYSPARK_PYTHON:", os.environ.get("PYSPARK_PYTHON"))
print("HADOOP_HOME:", os.environ.get("HADOOP_HOME"))

spark = SparkSession.builder \
    .appName("pyspark_ylep") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.sql.autoBroadcastJoinThreshold", "64MB") \
    .config("spark.local.dir", "D:/spark_tmp") \
    .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true") \
    .config("spark.python.worker.faulthandler.enabled", "true") \
    .config("spark.sql.execution.pythonUDF.arrow.enabled", "false") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1")\
    .getOrCreate()



PYSPARK_PYTHON: c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe
HADOOP_HOME: C:\hadoop


c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
df_read_business_data = spark.read.json("D:\\A_Data_engineer\\Project_Yelp\\Yelp JSON\\yelp_academic_dataset_business.json")

In [4]:
df_read_dim_business_data  = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_business")

In [5]:
df_read_dim_business_hours = spark.read.parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_business_hours")

In [6]:
df_hours = df_read_business_data.select(F.col("hours"),F.col("business_id"))

In [ ]:
def parse_schedule_entry(text):
   if text is None:
      return ["Not Specified"],["Not Specified"]  ,["Not Specified"]     

   operating_hours = str(text).split(",")
   start  = []
   end = []
   day =[]
   for schedule in operating_hours:
      start_hour="" 
      end_hour="" 
      count = 0

      clean_day_name = schedule.lower().strip()

      if "none" in clean_day_name or  "null" in clean_day_name or (
      "friday" not in clean_day_name and  
      "saturday" not in clean_day_name and 
      "sunday" not in clean_day_name and 
      "monday" not in clean_day_name and
      "tuesday" not in clean_day_name and
      "wednesday" not in clean_day_name and
      "thursday" not in clean_day_name  ) :
         continue
            
      for slot in schedule:
         if  slot == "-":
            count = 1 

         if slot == ":" and count == 1 :
            end_hour = end_hour + ":"

         if slot == ":" and count == 0 :
            start_hour = start_hour + ":"

         if  slot.isdigit() and count == 1:  
            end_hour=end_hour + slot

         if slot.isdigit() and count == 0:

            start_hour=start_hour + slot

      if not start_hour  or not end_hour :
         continue
   
      if "friday" in clean_day_name and "friday" not in day:
         day.append("friday")
        
      if "saturday" in clean_day_name and "saturday" not in day:
         day.append("saturday")
        
      if "sunday" in clean_day_name and "sunday" not in day:
         day.append("sunday")  
        
      if "monday" in clean_day_name and "monday" not in day:
         day.append("monday")
        
      if "tuesday" in clean_day_name and "tuesday" not in day:
         day.append("tuesday")
           
      if "wednesday" in clean_day_name and "wednesday" not in day:
         day.append("wednesday")
          
      if "thursday" in clean_day_name and "thursday" not in day:
         day.append("thursday")
           
      try:
         date_object = datetime.strptime(start_hour, '%H:%M').time()
         start_time_12h = date_object.strftime("%I:%M %p")
      except ValueError:
         start_time_12h = "Time Not Set"

      try:
         date_object = datetime.strptime(end_hour, '%H:%M').time()
         end_time_12h = date_object.strftime("%I:%M %p")
      except ValueError:
         end_time_12h = "Time Not Set"

      if start_time_12h in  "Time Not Set" or end_time_12h in "Time Not Set" :
        start_time_12h = "Time Not Set" 
        end_time_12h = "Time Not Set"  
   
      start.append(start_time_12h)
      end.append(end_time_12h )
      



   if "friday" not in day:
      day.append("friday")
      start.append("Time Not Set")
      end.append("Time Not Set")

   if "saturday" not in day:
      day.append("saturday")
      start.append("Time Not Set")
      end.append("Time Not Set")

   if "sunday" not in day:
      day.append("sunday")  
      start.append("Time Not Set")
      end.append("Time Not Set")

   if "monday" not in day:
      day.append("monday")
      start.append("Time Not Set")
      end.append("Time Not Set")
   if "tuesday" not in day:
      day.append("tuesday")   
      start.append("Time Not Set")
      end.append("Time Not Set")
   if "wednesday" not in day:
      day.append("wednesday")  
      start.append("Time Not Set")
      end.append("Time Not Set")
   if "thursday" not in day:
      day.append("thursday") 
      start.append("Time Not Set")
      end.append("Time Not Set")
   return day , start , end   


schema = StructType([
    StructField("day_name", ArrayType(StringType()), True), 
    StructField("shift_start_hour", ArrayType(StringType()), True),
    StructField("shift_end_hour", ArrayType(StringType()), True),  
])

parse_schedule_entry_udf = F.udf(parse_schedule_entry, schema)

df_elite = (
    df_hours.withColumn("new_eilte", parse_schedule_entry_udf(F.col("hours")))
      .withColumn("day_name", F.col("new_eilte.day_name"))
      .withColumn("shift_start_hour", F.col("new_eilte.shift_start_hour"))
      .withColumn("shift_end_hour", (F.col("new_eilte.shift_end_hour")))
).drop(F.col("new_eilte")).cache()  

df_elite.show(n=1000,truncate=False)

In [ ]:
df_explode = (
    df_elite.withColumn("zipped", F.arrays_zip("day_name", "shift_start_hour", "shift_end_hour"))
      .withColumn("zipped", F.explode(F.col("zipped")))
      .select(
         F.col("business_id"),  # or whatever your identifying columns are
          F.col("zipped.day_name").alias("day_name"),
          F.col("zipped.shift_start_hour").alias("shift_start_hour"),
          F.col("zipped.shift_end_hour").alias("shift_end_hour"),
      ).distinct()
)
df_explode.show(truncate=False)

In [12]:
# Join source and target on ID
join_condition =  ((df_explode["day_name"] == df_read_dim_business_hours["day_name"]  )
                 & (df_explode["shift_start_hour"] == df_read_dim_business_hours["shift_start_hour"]) 
                 & (df_explode["shift_end_hour"] == df_read_dim_business_hours["shift_end_hour"] ) )




merged_join= df_explode.join(df_read_dim_business_hours, join_condition, "inner")

In [ ]:
merged_join.orderBy(F.col("business_id")).show(n=1000,truncate=False)

In [ ]:
df_final = merged_join.select(F.col("business_id") ,F.col("business_hours_pk_sk"))

In [ ]:
df_final.show(truncate=False)

+----------------------+--------------------+
|business_id           |business_hours_pk_sk|
+----------------------+--------------------+
|iaLaRfHUysXkKd1iNCTWbw|5201                |
|2IVBu48J_PvBLuEy2Ubi5A|5201                |
|iY8FpFaR9jVyhKQHQ00Tug|5201                |
|1R_PKvh6D0g0UKDInU5ulg|5201                |
|7UqYL6gIuHyBy6ryMtHM8g|5201                |
|xdw_l_qzMSGRQ1j9m18qpQ|5201                |
|cGSzV7rhpySDoc_ONBZjEw|5201                |
|Qo_ka_tfXxrEpa_QLE0rdQ|5201                |
|-0YvX6VK5S0u_hbQFYq10g|5201                |
|BBEP5WboQM-42ceJTwh58w|38                  |
|eLrLn5vr05GEOhJgREUpmw|38                  |
|h9C0Gub8prrtAwLDSbsAfQ|38                  |
|owQgiBoTu821TG8d-w5shg|38                  |
|scrFzCIy6TTHu1nnYsJdFg|38                  |
|2UXAIQZWJACELFmyHFCHaw|38                  |
|UdNx91_fhJthJhhzSV4Vkg|38                  |
|20Wicri3Ij4TWIv1FJ3d2g|38                  |
|7siEI8YBEnL4tZOnPUg_sA|38                  |
|ZUutpEtpiEAzKgCNJDsy1g|38        

In [79]:
#df_read_dim_business_hours.select(max(F.lit(("business_hours_pk_sk")))).show()

#df_read_dim_business_hours.agg({"business_hours_pk_sk": "max"}).show()
mx =  df_read_dim_business_hours.agg(F.max("business_hours_pk_sk").alias("max_business_hours_pk_sk")).first()

In [80]:
# Source - https://stackoverflow.com/a/73653726
# Posted by bzu, modified by community. See post 'Timeline' for change history
# Retrieved 2026-09-22, License - CC BY-SA 4.0

#n = df_a.select(F.max('unique_id').alias('max_n')).first().max_n
#df_b = df_b.withColumn("unique_id", F.monotonically_increasing_id() + F.lit(mx + 1))


In [86]:
df_read_dim_business_hours.filter(F.col("shift_start_hour").contains("1:19 PM") ).show()

+--------------------+--------+----------------+--------------+
|business_hours_pk_sk|day_name|shift_start_hour|shift_end_hour|
+--------------------+--------+----------------+--------------+
+--------------------+--------+----------------+--------------+



In [ ]:
# Join source and target on ID
join_condition =  ((df_explode["day_name"] == df_read_dim_business_hours["day_name"]  )
                 & (df_explode["shift_start_hour"] == df_read_dim_business_hours["shift_start_hour"]) 
                 & (df_explode["shift_end_hour"] == df_read_dim_business_hours["shift_end_hour"] ) )




merged_df = df_explode.join(df_read_dim_business_hours, join_condition, "LeftAnti")



In [ ]:
#merged_df_2.write.mode("append").parquet("D:/A_Data_engineer/Project_Yelp/store_dimesnion_user/dim_business_hours")

In [ ]:
merged_join.show(truncate=False)


+--------------------+--------+----------------+--------------+--------+----------------+--------------+--------------------+
|         business_id|day_name|shift_start_hour|shift_end_hour|day_name|shift_start_hour|shift_end_hour|business_hours_pk_sk|
+--------------------+--------+----------------+--------------+--------+----------------+--------------+--------------------+
|iaLaRfHUysXkKd1iN...|  friday|        01:00 PM|      05:30 PM|  friday|        01:00 PM|      05:30 PM|                5201|
|2IVBu48J_PvBLuEy2...|  friday|        01:00 PM|      05:30 PM|  friday|        01:00 PM|      05:30 PM|                5201|
|iY8FpFaR9jVyhKQHQ...|  friday|        01:00 PM|      05:30 PM|  friday|        01:00 PM|      05:30 PM|                5201|
|1R_PKvh6D0g0UKDIn...|  friday|        01:00 PM|      05:30 PM|  friday|        01:00 PM|      05:30 PM|                5201|
|7UqYL6gIuHyBy6ryM...|  friday|        01:00 PM|      05:30 PM|  friday|        01:00 PM|      05:30 PM|              

In [15]:
type(mx)

pyspark.sql.types.Row

In [16]:
mx =  df_read_dim_business_hours.agg(F.max("business_hours_pk_sk").alias("max_business_hours_pk_sk")).first().max_business_hours_pk_sk
merged_df_2 = merged_df.withColumn("business_hours_pk_sk", F.monotonically_increasing_id() + F.lit(mx + 1))
merged_df_2.show()


+---------+----------------+--------------+--------------------+
| day_name|shift_start_hour|shift_end_hour|business_hours_pk_sk|
+---------+----------------+--------------+--------------------+
|   friday|        01:09 AM|      08:05 AM|                9871|
|   monday|        01:11 AM|      03:58 AM|                9872|
| saturday|        04:08 AM|      12:58 PM|                9873|
| thursday|        01:16 AM|      08:40 AM|                9874|
|  tuesday|        01:15 AM|      04:58 AM|                9875|
|wednesday|        01:19 AM|      08:50 AM|                9876|
+---------+----------------+--------------+--------------------+



In [84]:
import pyspark.sql.functions as F

# Path setup
target_parquet_path = "D:/data/dim_user"

# 1. Load Existing Target & New Batch
df_target = spark.read.parquet(target_parquet_path)
# Target schema: [user_sk, user_id, city, valid_from, valid_to, is_current]

df_incoming = spark.read.parquet("D:/data/raw_user_updates")
# Incoming schema: [user_id, city, update_time]

current_time = F.col("update_time")

# 2. Join Incoming with Current Active Target Records
df_active_target = df_target.filter(F.col("is_current") == True)

df_joined = df_active_target.alias("tgt").join(
    df_incoming.alias("inc"),
    on="user_id",
    how="right"
)

# ---------------------------------------------------------
# A. Unchanged / Historical Rows (untouched)
# ---------------------------------------------------------
df_untouched_history = df_target.filter(F.col("is_current") == False)

# ---------------------------------------------------------
# B. Records to Expire (Existing active records that changed)
# ---------------------------------------------------------
df_expired = (
    df_joined.filter(
        F.col("tgt.user_id").isNotNull() & 
        (F.col("tgt.city") != F.col("inc.city"))
    )
    .select(
        F.col("tgt.user_sk"),
        F.col("tgt.user_id"),
        F.col("tgt.city"),
        F.col("tgt.valid_from"),
        F.col("inc.update_time").alias("valid_to"),
        F.lit(False).alias("is_current")
    )
)

# ---------------------------------------------------------
# C. Records that stay Active (Existing active with no changes)
# ---------------------------------------------------------
df_unmodified = (
    df_active_target.join(df_incoming, on="user_id", how="left_anti")
)

# ---------------------------------------------------------
# D. New Active Records (Brand new keys + Updated versions)
# ---------------------------------------------------------
# Filter for records that are brand new OR have changed values
df_new_active = (
    df_joined.filter(
        F.col("tgt.user_id").isNull() | 
        (F.col("tgt.city") != F.col("inc.city"))
    )
    .select(
        # Deterministic Surrogate Key generation via Hash
        F.sha2(F.concat_ws("||", F.col("inc.user_id"), F.col("inc.update_time")), 256).alias("user_sk"),
        F.col("inc.user_id"),
        F.col("inc.city"),
        F.col("inc.update_time").alias("valid_from"),
        F.to_timestamp(F.lit("9999-12-31 23:59:59")).alias("valid_to"),
        F.lit(True).alias("is_current")
    )
)

# 3. Union All Slices and Overwrite Parquet Target
df_final_scd2 = (
    df_untouched_history
    .unionByName(df_expired)
    .unionByName(df_unmodified)
    .unionByName(df_new_active)
)

df_final_scd2.write.mode("overwrite").parquet(target_parquet_path)

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/D:/data/dim_user. SQLSTATE: 42K03